# KLUE-RoBERTa 데일리 분류기 Fine-tuning

## 평가 구조
1. **데이터 분할**: Train 70% / Val 15% / Test 15%
2. **학습 중 평가**: Validation Accuracy 모니터링
3. **Early Stopping**: 2회 연속 개선 없으면 자동 종료
4. **최종 평가**: Before/After 정확도 + Classification Report

## 분류 카테고리
- **YESTERDAY**: 어제 한 일
- **TODAY**: 오늘 할 일
- **BLOCKER**: 블로커/이슈

In [ ]:
# ===== 0. 환경 설정 =====
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

import torch
import random
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# 재현성
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

In [ ]:
# 라벨 정의
LABELS = ["YESTERDAY", "TODAY", "BLOCKER"]
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
ID2LABEL = {i: label for i, label in enumerate(LABELS)}

print(f"분류 카테고리: {LABELS}")

---
## 1. 데이터 생성 및 분할 (70/15/15)

In [ ]:
# 템플릿 기반 합성 데이터 생성
TEMPLATES = {
    "YESTERDAY": [
        "어제 {task} 완료했습니다", "어제 {task} 끝냈어요", "{task} 작업 마무리했습니다",
        "{task} 개발 완료했어요", "어제는 {task} 진행했습니다", "{task} 구현 끝났습니다",
        "어제 {task} 처리했어요", "{task} 배포까지 완료했습니다", "PR 올리고 {task} 머지했어요",
        "{task} 테스트 통과했습니다", "어제 {task} 리뷰받고 수정했어요", "{task} 코드 작성 완료",
        "지난번 말씀드린 {task} 다 했습니다", "{task} 기능 추가 완료했어요",
    ],
    "TODAY": [
        "오늘 {task} 할 예정입니다", "{task} 작업 시작하려고 해요", "오늘은 {task} 진행할게요",
        "{task} 구현해야 합니다", "오늘 {task} 개발할 예정이에요", "{task} 마무리하려고 합니다",
        "오늘 계획은 {task} 입니다", "{task} 작업 들어갈게요", "오늘 {task} 처리해야 해요",
        "{task} 이어서 진행하겠습니다", "오늘 목표는 {task} 완료입니다",
    ],
    "BLOCKER": [
        "{issue} 문제로 진행이 안 돼요", "{issue} 때문에 막혀있습니다", "{issue} 해결이 필요해요",
        "{issue} 이슈가 있습니다", "{issue} 없이는 진행 불가해요", "{issue} 기다리고 있습니다",
        "{issue} 확인이 필요합니다", "{issue} 권한이 없어서 못해요", "{issue} 충돌이 발생했습니다",
        "{issue} 나와야 작업 가능해요", "{issue} 안 되면 멈춰야 합니다",
    ]
}

TASKS = [
    "로그인 API", "회원가입", "결제 모듈", "알림 기능", "대시보드", "검색 기능", 
    "파일 업로드", "채팅 기능", "푸시 알림", "OAuth 연동", "배포 스크립트", 
    "DB 마이그레이션", "API 문서", "테스트 코드", "코드 리뷰", "버그 수정",
    "성능 최적화", "UI 개선", "에러 핸들링", "로깅 추가"
]

ISSUES = [
    "AWS 권한", "디자인 시안", "백엔드 API", "서버 접속", "라이브러리 버전", 
    "인증 토큰", "DB 연결", "빌드 에러", "테스트 환경", "코드 충돌",
    "VPN 연결", "계정 권한", "외부 API", "인프라 설정", "보안 정책"
]

def generate_data(n_per_class=500):
    data = []
    for label, templates in TEMPLATES.items():
        items = TASKS if label != "BLOCKER" else ISSUES
        key = "task" if label != "BLOCKER" else "issue"
        for _ in range(n_per_class):
            text = random.choice(templates).format(**{key: random.choice(items)})
            data.append({"text": text, "label": label})
    random.shuffle(data)
    return data

# 데이터 생성
all_data = generate_data(500)  # 클래스당 500개 = 총 1500개
print(f"전체 데이터: {len(all_data)}개")

# ===== 핵심: Train/Val/Test 분할 =====
train_data, temp_data = train_test_split(all_data, test_size=0.3, random_state=42, stratify=[d['label'] for d in all_data])
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42, stratify=[d['label'] for d in temp_data])

print(f"\n[데이터 분할]")
print(f"  Train:      {len(train_data):>5}개 (70%)")
print(f"  Validation: {len(val_data):>5}개 (15%)")
print(f"  Test:       {len(test_data):>5}개 (15%)")

# 분포 확인
from collections import Counter
print(f"\n[Train 라벨 분포]")
for label, count in Counter([d['label'] for d in train_data]).items():
    print(f"  {label}: {count}개")

---
## 2. 고정 테스트 샘플 (Before/After 비교용)

In [ ]:
# Before/After 비교용 고정 샘플
FIXED_TEST_SAMPLES = [
    ("어제 로그인 API 개발 완료했습니다", "YESTERDAY"),
    ("PR 리뷰하고 머지했어요", "YESTERDAY"),
    ("DB 스키마 설계 끝냈습니다", "YESTERDAY"),
    ("테스트 코드 작성 완료했어요", "YESTERDAY"),
    ("오늘 회원가입 기능 구현할 예정입니다", "TODAY"),
    ("배포 파이프라인 설정할게요", "TODAY"),
    ("코드 리팩토링 진행하려고 합니다", "TODAY"),
    ("API 문서 작성해야 해요", "TODAY"),
    ("AWS 권한 문제로 배포가 안 돼요", "BLOCKER"),
    ("디자인 시안이 아직 안 나왔습니다", "BLOCKER"),
    ("백엔드 API가 완성되어야 진행 가능해요", "BLOCKER"),
    ("테스트 서버 접속이 안 됩니다", "BLOCKER"),
]

print(f"고정 테스트 샘플: {len(FIXED_TEST_SAMPLES)}개")

---
## 3. [BEFORE] Zero-shot 베이스라인

In [ ]:
from transformers import pipeline

print("[BEFORE] Zero-shot 분류기 로딩...")
zero_shot = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7",
    device=0
)

KOREAN_LABELS = ["어제 한 일", "오늘 할 일", "블로커"]
KOREAN_TO_LABEL = {"어제 한 일": "YESTERDAY", "오늘 할 일": "TODAY", "블로커": "BLOCKER"}

print("\n" + "=" * 60)
print("[BEFORE] Zero-shot 분류 결과")
print("=" * 60)

before_preds = []
before_gts = []

for text, gt in FIXED_TEST_SAMPLES:
    result = zero_shot(text, KOREAN_LABELS)
    pred = KOREAN_TO_LABEL[result["labels"][0]]
    before_preds.append(pred)
    before_gts.append(gt)
    status = "O" if pred == gt else "X"
    print(f"{status} [{gt:10}] {text[:35]:35} -> {pred}")

before_acc = accuracy_score(before_gts, before_preds)
print(f"\n[BEFORE] 정확도: {before_acc:.1%}")

In [ ]:
# 메모리 해제
del zero_shot
torch.cuda.empty_cache()
print("Zero-shot 모델 메모리 해제")

---
## 4. 모델 로드 및 데이터셋 준비

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import Dataset

MODEL_NAME = "klue/roberta-large"

print(f"모델 로딩: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID
).to("cuda")

print(f"모델 파라미터: {model.num_parameters():,}")
print(f"VRAM: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")

In [ ]:
def preprocess(examples):
    tokenized = tokenizer(
        examples["text"], 
        truncation=True, 
        padding="max_length", 
        max_length=128
    )
    tokenized["labels"] = [LABEL2ID[l] for l in examples["label"]]
    return tokenized

# 데이터셋 변환
train_dataset = Dataset.from_list(train_data).map(preprocess, batched=True)
val_dataset = Dataset.from_list(val_data).map(preprocess, batched=True)
test_dataset = Dataset.from_list(test_data).map(preprocess, batched=True)

# 포맷 설정
for ds in [train_dataset, val_dataset, test_dataset]:
    ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

---
## 5. Trainer 설정 (Early Stopping 포함)

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

training_args = TrainingArguments(
    output_dir="./klue_roberta_classifier",
    
    # === 학습 설정 ===
    num_train_epochs=10,              # 최대 10 에포크
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    
    # === 최적화 ===
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    
    # === 핵심: 평가 설정 ===
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,           # accuracy는 높을수록 좋음
    
    # === 로깅 ===
    logging_steps=50,
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,         # ← Validation 데이터셋!
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=2)  # 2회 연속 개선 없으면 종료
    ],
)

print("[Trainer 설정]")
print(f"  - Max epochs: {training_args.num_train_epochs}")
print(f"  - Eval: 매 에포크")
print(f"  - Early stopping: patience=2")
print(f"  - Best model: accuracy 기준 자동 선택")

---
## 6. 학습 실행

In [ ]:
print("=" * 60)
print("학습 시작")
print("=" * 60)
print("Validation Accuracy가 2회 연속 개선되지 않으면 자동 종료됩니다.\n")

train_result = trainer.train()

print("\n" + "=" * 60)
print("학습 완료!")
print("=" * 60)

---
## 7. 학습 곡선 시각화

In [ ]:
import matplotlib.pyplot as plt

logs = trainer.state.log_history

# Accuracy 추출
eval_acc = [(i+1, x['eval_accuracy']) for i, x in enumerate([l for l in logs if 'eval_accuracy' in l])]
train_loss = [(x['epoch'], x['loss']) for x in logs if 'loss' in x and 'eval' not in str(x)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Train Loss
if train_loss:
    epochs, losses = zip(*train_loss)
    axes[0].plot(epochs, losses, marker='o')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Train Loss')
    axes[0].set_title('Training Loss')
    axes[0].grid(True, alpha=0.3)

# Validation Accuracy
if eval_acc:
    epochs, accs = zip(*eval_acc)
    axes[1].plot(epochs, accs, marker='o', color='green')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Validation Accuracy')
    axes[1].set_title('Validation Accuracy')
    axes[1].grid(True, alpha=0.3)
    
    # 최고점 표시
    best_idx = np.argmax(accs)
    axes[1].axvline(x=epochs[best_idx], color='red', linestyle='--', alpha=0.5)
    axes[1].annotate(f'Best: {accs[best_idx]:.1%}', 
                     xy=(epochs[best_idx], accs[best_idx]),
                     xytext=(5, -10), textcoords='offset points')

plt.tight_layout()
plt.savefig('classifier_training_curve.png', dpi=150)
plt.show()

if eval_acc:
    print(f"\n[최적 에포크]")
    print(f"  Epoch: {epochs[best_idx]}")
    print(f"  Val Accuracy: {accs[best_idx]:.1%}")

---
## 8. [AFTER] 파인튜닝 후 테스트

In [ ]:
# 파인튜닝된 모델로 파이프라인 생성
finetuned_pipe = pipeline(
    "text-classification", 
    model=model, 
    tokenizer=tokenizer, 
    device=0
)

print("=" * 60)
print("[AFTER] 파인튜닝 모델 분류 결과")
print("=" * 60)

after_preds = []
after_gts = []

for text, gt in FIXED_TEST_SAMPLES:
    result = finetuned_pipe(text)[0]
    pred = result["label"]
    conf = result["score"]
    after_preds.append(pred)
    after_gts.append(gt)
    status = "O" if pred == gt else "X"
    print(f"{status} [{gt:10}] {text[:35]:35} -> {pred} ({conf:.1%})")

after_acc = accuracy_score(after_gts, after_preds)
print(f"\n[AFTER] 정확도: {after_acc:.1%}")

---
## 9. Test 데이터셋 최종 평가

In [ ]:
print("=" * 60)
print("[TEST SET] 최종 평가")
print("=" * 60)

# Test 데이터셋 평가
test_results = trainer.evaluate(test_dataset)
print(f"\nTest Accuracy: {test_results['eval_accuracy']:.1%}")

# 상세 리포트
test_preds = []
test_labels = []

for item in test_data:
    result = finetuned_pipe(item['text'])[0]
    test_preds.append(result['label'])
    test_labels.append(item['label'])

print("\n[Classification Report]")
print(classification_report(test_labels, test_preds, target_names=LABELS))

In [ ]:
# Confusion Matrix 시각화
import seaborn as sns

cm = confusion_matrix(test_labels, test_preds, labels=LABELS)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=LABELS, yticklabels=LABELS)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

---
## 10. Before/After 비교 요약

In [ ]:
print("=" * 60)
print("[SUMMARY] Before vs After 비교")
print("=" * 60)

print(f"\n| 구분 | 정확도 |")
print(f"|------|--------|")
print(f"| Before (Zero-shot) | {before_acc:.1%} |")
print(f"| After (Fine-tuned) | {after_acc:.1%} |")
print(f"| Test Set           | {test_results['eval_accuracy']:.1%} |")

improvement = (after_acc - before_acc) / before_acc * 100 if before_acc > 0 else 0
print(f"\n개선율: {improvement:+.1f}%")

---
## 11. 모델 저장

In [ ]:
SAVE_PATH = "./klue_roberta_classifier_final"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f"모델 저장 완료: {SAVE_PATH}")
!ls -lh {SAVE_PATH}

---
## 12. 최종 리포트

In [ ]:
print("\n" + "=" * 60)
print("최종 학습 리포트")
print("=" * 60)

print(f"\n[모델]")
print(f"  Base: {MODEL_NAME}")
print(f"  파라미터: {model.num_parameters():,}")
print(f"  분류 클래스: {LABELS}")

print(f"\n[데이터]")
print(f"  Train: {len(train_data)}개 (70%)")
print(f"  Val:   {len(val_data)}개 (15%)")
print(f"  Test:  {len(test_data)}개 (15%)")

print(f"\n[성능]")
print(f"  Before (Zero-shot): {before_acc:.1%}")
print(f"  After (Fine-tuned): {after_acc:.1%}")
print(f"  Test Set:           {test_results['eval_accuracy']:.1%}")

print(f"\n[저장]")
print(f"  모델: {SAVE_PATH}")
print(f"  그래프: classifier_training_curve.png")
print(f"  혼동행렬: confusion_matrix.png")

print("\n" + "=" * 60)
print("완료!")
print("=" * 60)